# GridMind GR — Load Feature Engineering

This notebook builds the final v1 feature set for day-ahead system load forecasting.

The production model intentionally uses a small six-feature set: calendar context plus load values from 24, 48 and 168 hours earlier.


In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../src")

from gridmind.data.queries import load_historical_data


In [ ]:
MIN_VALID_LOAD_MWH = 500

load = load_historical_data()
load["timestamp"] = pd.to_datetime(load["timestamp"])
load = load.sort_values("timestamp").reset_index(drop=True)

load.loc[
    load["load_mwh"] < MIN_VALID_LOAD_MWH,
    "load_mwh",
] = np.nan

print(f"Rows: {len(load):,}")
print(f"Invalid load values after cleaning: {load['load_mwh'].isna().sum()}")
print(
    f"Period: {load['timestamp'].min()} "
    f"-> {load['timestamp'].max()}"
)


## Final v1 features

`load_lag_24`, `load_lag_48` and `load_lag_168` are compatible with day-ahead forecasting because they only use load values that are already known before the forecast day.


In [ ]:
FEATURE_COLS = [
    "hour",
    "day_of_week",
    "is_weekend",
    "load_lag_24",
    "load_lag_48",
    "load_lag_168",
]

load["target"] = load["load_mwh"]
load["hour"] = load["timestamp"].dt.hour
load["day_of_week"] = load["timestamp"].dt.dayofweek
load["is_weekend"] = (load["day_of_week"] >= 5).astype(int)

load["load_lag_24"] = (load["timestamp"] - pd.Timedelta(hours=24)).map(load.set_index("timestamp")["load_mwh"])
load["load_lag_48"] = (load["timestamp"] - pd.Timedelta(hours=48)).map(load.set_index("timestamp")["load_mwh"])
load["load_lag_168"] = (load["timestamp"] - pd.Timedelta(hours=168)).map(load.set_index("timestamp")["load_mwh"])

model_data = (
    load[["timestamp", "target", *FEATURE_COLS]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)

model_data.head()


## Final checks


In [ ]:
print(f"Shape: {model_data.shape}")
print(f"Features: {FEATURE_COLS}")
print(f"Missing values: {model_data.isna().sum().sum()}")
print(f"Duplicate timestamps: {model_data['timestamp'].duplicated().sum()}")
print(f"Target range: {model_data['target'].min():.0f} -> {model_data['target'].max():.0f} MWh")
print(
    f"Model period: {model_data['timestamp'].min()} "
    f"-> {model_data['timestamp'].max()}"
)


In [ ]:
model_data[
    ["target", "load_lag_24", "load_lag_48", "load_lag_168"]
].describe()


### Feature engineering conclusion

The v1 load model uses exactly six features. Experimental cyclical, rolling-window and short-lag features were removed from the final pipeline to keep the production model simple and aligned with the day-ahead forecasting setup.

No intermediate CSV is produced; PostgreSQL remains the source of truth.
